In [45]:
! pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [46]:
import pandas as pd 
import numpy as np
import kagglehub
import os

In [47]:
# Descargar la última version del dataset de Kaggle
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

# Revisar que archivos contiene
print(os.listdir(path))

Path to dataset files: C:\Users\Ivanna\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Comprensión de los datos

In [48]:
import pandas as pd
import os

customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(path, "olist_geolocation_dataset.csv"))
orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv"))
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(
    os.path.join(path, "product_category_name_translation.csv")
)

## Dimensión de las bases de datos

In [49]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")

Customers: 99441 filas, 5 columnas
Geolocation: 1000163 filas, 5 columnas
Orders: 99441 filas, 8 columnas
Order Items: 112650 filas, 7 columnas
Order Payments: 103886 filas, 5 columnas
Order Reviews: 99224 filas, 7 columnas
Products: 32951 filas, 9 columnas
Sellers: 3095 filas, 4 columnas
Category Translation: 71 filas, 2 columnas


## Duplicados

In [50]:
for nombre, df in datasets.items():
    print(f"{nombre}: {df.duplicated().sum()} duplicados")

Customers: 0 duplicados
Geolocation: 261831 duplicados
Orders: 0 duplicados
Order Items: 0 duplicados
Order Payments: 0 duplicados
Order Reviews: 0 duplicados
Products: 0 duplicados
Sellers: 0 duplicados
Category Translation: 0 duplicados


En este caso estuvimos viendo los duplicados en las base de datos en dónde pudimos encontrar que la única con estos fué Geolocation con 261831 duplicados, pero estos se estarán viendo más adelante en la parte de la exploración de los datos. 

## Verificar las llaves primarias

In [51]:
print(customers["customer_id"].is_unique)
print(orders["order_id"].is_unique)
print(products["product_id"].is_unique)
print(sellers["seller_id"].is_unique)
print(category_translation["product_category_name"].is_unique)

True
True
True
True
True


Cómo podemos ver las llaves primarias de las bases de datos son las siguientes:

* customer_id para la base de datos Customers
* order_id para la base de datos Orders
* product_id para la base de datos Products
* geolocation_id para la base de datos Geolocation. 

La cuales nos ayudarán a conectar las bases de datos de acuerdo a nuestro objetivo de negocio y poder llegar a una solución.

## Revisar los tipos de datos

In [52]:
for nombre, df in datasets.items():
    print(f"\n{nombre}")
    print(df.dtypes)


Customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Orders
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Order Items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype:

Se verificaron los tipos de datos de las nueve tablas del conjunto de datos. Los identificadores se encuentran almacenados como variables de tipo object, mientras que las variables numéricas presentan tipos int64 y float64, lo cual es consistente con la naturaleza de la variable. Se identificó que las variables correspondientes a fechas y horas en las tablas orders, order_items y order_reviews se encuentran almacenadas como object; por tanto, durante la fase de preparación de los datos se convertirán al tipo datetime para facilitar el análisis temporal. No se identificaron inconsistencias relevantes en los demás tipos de datos.

## Completitud de los datos

### Calcular valores nulos

In [53]:
def completitud(df):
    reporte = pd.DataFrame({
        "Valores no nulos": df.notnull().sum(),
        "Valores nulos": df.isnull().sum(),
        "Porcentaje de nulos (%)": round(df.isnull().mean() * 100, 2)
    })

    return reporte.sort_values("Porcentaje de nulos (%)", ascending=False)

In [54]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"\n===== {nombre} =====")
    print(completitud(df))


===== Customers =====
                          Valores no nulos  Valores nulos  \
customer_id                          99441              0   
customer_unique_id                   99441              0   
customer_zip_code_prefix             99441              0   
customer_city                        99441              0   
customer_state                       99441              0   

                          Porcentaje de nulos (%)  
customer_id                                   0.0  
customer_unique_id                            0.0  
customer_zip_code_prefix                      0.0  
customer_city                                 0.0  
customer_state                                0.0  

===== Geolocation =====
                             Valores no nulos  Valores nulos  \
geolocation_zip_code_prefix           1000163              0   
geolocation_lat                       1000163              0   
geolocation_lng                       1000163              0   
geolocation_city 

En este caso evaluamos la completitud de las nueve tablas del conjunto de datos mediante la cantidad de valores nulos por variable.

Los principales hallazgos que pudimos encontrar fueron los siguientes:

* Customers: presenta un 100 % de completitud, ya que no se identificaron valores faltantes en ninguna de sus variables.
* Geolocation: todas las variables presentan un 100 % de completitud.
* Orders: se identificaron valores faltantes en las columnas order_delivered_customer_date (2,98 %), order_delivered_carrier_date (1,79 %) y order_approved_at (0,16 %). Estos valores pueden corresponder a pedidos que aún no habían sido aprobados o entregados al momento del registro, por lo que no necesariamente representan errores en los datos.
* Order Items: todas las variables presentan un 100 % de completitud.
* Order Payments: no se encontraron valores faltantes.
* Order Reviews: se observó un alto porcentaje de valores faltantes en review_comment_title (88,34 %) y review_comment_message (58,70 %). Esto es consistente con el hecho de que los comentarios textuales son opcionales y muchos clientes únicamente asignan una calificación sin escribir una reseña.
* roducts: se identificaron valores faltantes en product_category_name, product_name_lenght, product_description_lenght y product_photos_qty (1,85 % cada una), así como un porcentaje mínimo (0,01 %) en las variables de peso y dimensiones.
* Sellers: presenta un 100 % de completitud.
* Category Translation: no presenta valores faltantes.

En general, el conjunto de datos presenta un alto nivel de completitud. Los valores faltantes se concentran principalmente en las tablas order_reviews, debido a la naturaleza opcional de los comentarios de los clientes, y products, donde el porcentaje de datos faltantes es bajo. Estos casos serán tratados durante la fase de preparación de los datos.

##  Consistencia de los datos



### Distribución de los datos para establecer los supuestos de rango esperado


In [55]:
print(products.describe())

       product_name_lenght  product_description_lenght  product_photos_qty  \
count         32341.000000                32341.000000        32341.000000   
mean             48.476949                  771.495285            2.188986   
std              10.245741                  635.115225            1.736766   
min               5.000000                    4.000000            1.000000   
25%              42.000000                  339.000000            1.000000   
50%              51.000000                  595.000000            1.000000   
75%              57.000000                  972.000000            3.000000   
max              76.000000                 3992.000000           20.000000   

       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000           7.0

In [56]:
print(order_items.describe())

       order_item_id          price  freight_value
count  112650.000000  112650.000000  112650.000000
mean        1.197834     120.653739      19.990320
std         0.705124     183.633928      15.806405
min         1.000000       0.850000       0.000000
25%         1.000000      39.900000      13.080000
50%         1.000000      74.990000      16.260000
75%         1.000000     134.900000      21.150000
max        21.000000    6735.000000     409.680000


In [57]:
print(order_payments.describe())

       payment_sequential  payment_installments  payment_value
count       103886.000000         103886.000000  103886.000000
mean             1.092679              2.853349     154.100380
std              0.706584              2.687051     217.494064
min              1.000000              0.000000       0.000000
25%              1.000000              1.000000      56.790000
50%              1.000000              1.000000     100.000000
75%              1.000000              4.000000     171.837500
max             29.000000             24.000000   13664.080000


In [58]:
print(order_reviews.describe())

       review_score
count  99224.000000
mean       4.086421
std        1.347579
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000


In [59]:
print(geolocation.describe())

       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng
count                 1.000163e+06     1.000163e+06     1.000163e+06
mean                  3.657417e+04    -2.117615e+01    -4.639054e+01
std                   3.054934e+04     5.715866e+00     4.269748e+00
min                   1.001000e+03    -3.660537e+01    -1.014668e+02
25%                   1.107500e+04    -2.360355e+01    -4.857317e+01
50%                   2.653000e+04    -2.291938e+01    -4.663788e+01
75%                   6.350400e+04    -1.997962e+01    -4.376771e+01
max                   9.999000e+04     4.506593e+01     1.211054e+02


### **Supuestos de rango esperado**

| Variable               | Estadísticos observados   | Supuesto de consistencia                                                                                                                                                             | Resultado esperado                                                                                                            |
| ---------------------- | ------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------- |
| `price`                | Min = 0.85                | El precio debe ser **mayor o igual a 0**.                                                                                                                                            | No deberían existir precios negativos.                                                                                        |
| `freight_value`        | Min = 0.00                | El costo de envío debe ser **mayor o igual a 0**.                                                                                                                                    | Un valor de 0 puede corresponder a envíos gratuitos.                                                                          |
| `payment_installments` | Min = 0                   | El número de cuotas debe ser un **entero mayor o igual a 0**. Los registros con valor 0 serán revisados para determinar si corresponden a tipos de pago donde las cuotas no aplican. | No deberían existir valores negativos.                                                                                        |
| `payment_value`        | Min = 0                   | El valor del pago debe ser **mayor o igual a 0**.                                                                                                                                    | Los valores iguales a 0 serán revisados para confirmar si corresponden a pagos con cupones, descuentos u otros casos válidos. |
| `review_score`         | Min = 1, Max = 5          | La calificación debe estar entre **1 y 5**.                                                                                                                                          | Todos los registros deberían cumplir esta regla.                                                                              |
| `product_weight_g`     | Min = 0                   | El peso del producto debe ser **mayor que 0**. Los registros con peso igual a 0 serán revisados para determinar si corresponden a errores de captura o información faltante.         | No deberían existir pesos negativos; los valores 0 serán evaluados.                                                           |
| `product_length_cm`    | Min = 7                   | La longitud del producto debe ser **mayor que 0**.                                                                                                                                   | No se esperan inconsistencias.                                                                                                |
| `product_height_cm`    | Min = 2                   | La altura del producto debe ser **mayor que 0**.                                                                                                                                     | No se esperan inconsistencias.                                                                                                |
| `product_width_cm`     | Min = 6                   | El ancho del producto debe ser **mayor que 0**.                                                                                                                                      | No se esperan inconsistencias.                                                                                                |
| `product_photos_qty`   | Min = 1                   | La cantidad de fotografías debe ser **mayor o igual a 1**.                                                                                                                           | No se esperan inconsistencias.                                                                                                |
| `geolocation_lat`      | Min = -36.6, Max = 45.1   | La latitud debe estar entre **-90 y 90** grados.                                                                                                                                     | Todos los registros deberían cumplir esta regla.                                                                              |
| `geolocation_lng`      | Min = -101.4, Max = 121.1 | La longitud debe estar entre **-180 y 180** grados.                                                                                                                                  | Todos los registros deberían cumplir esta regla.                                                                              |


### Valores fuera del rango esperado

In [60]:
print(customers.columns)
print(geolocation.columns)
print(orders.columns)
print(order_items.columns)
print(order_payments.columns)
print(order_reviews.columns)
print(products.columns)
print(sellers.columns)
print(category_translation.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
Index(['product_id', 'prod

#### Comprobar inconsistencias

In [61]:
# Precios negativos
order_items[order_items["price"] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [62]:
order_items[order_items["freight_value"] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [63]:
order_payments[order_payments["payment_installments"] < 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [64]:
order_payments[order_payments["payment_value"] < 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [65]:
order_reviews[
    (order_reviews["review_score"] < 1) |
    (order_reviews["review_score"] > 5)
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [66]:
products[products["product_weight_g"] <= 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


Se definió como supuesto que el peso de un producto debe ser mayor que 0 g. La validación identificó 4 registros con peso igual a 0 g. Aunque estos productos cuentan con dimensiones, categoría y descripción registradas, un peso nulo resulta poco consistente para un producto físico. Por ello, estos casos se consideran posibles inconsistencias y serán tratados durante la fase de preparación de los datos.

In [67]:
(products["product_length_cm"] <= 0).sum()

(products["product_height_cm"] <= 0).sum()

(products["product_width_cm"] <= 0).sum()

np.int64(0)

In [68]:
(products["product_photos_qty"] < 1).sum()

np.int64(0)

In [69]:
(
    (geolocation["geolocation_lat"] < -90) |
    (geolocation["geolocation_lat"] > 90)
).sum()

np.int64(0)

In [70]:
(
    (geolocation["geolocation_lng"] < -180) |
    (geolocation["geolocation_lng"] > 180)
).sum()

np.int64(0)

Se validaron las variables numéricas de acuerdo con los supuestos de consistencia definidos previamente. No se identificaron precios, costos de envío, valores de pago ni calificaciones fuera de los rangos esperados. Sin embargo, se encontraron 2 productos con peso igual a 0, los cuales serán revisados durante la etapa de limpieza para determinar si corresponden a errores de captura o a información faltante.

## Consistencia temporal

In [71]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

- Revisar que la aprobación no puede ser antes de la compra

In [72]:
orders[
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


- El envío no puede ocurrir antes de la aprobación

In [73]:
print(orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
])

                               order_id                       customer_id  \
15     dcb36b511fcac050b97cd5c05de84dc3  3b6828a50ffe546942b7a473d70ac0fc   
64     688052146432ef8253587b930b01a06d  81e08b08e5ed4472008030d70327c71f   
199    58d4c4747ee059eeeb865b349b41f53a  1755fad7863475346bc6c3773fe055d3   
210    412fccb2b44a99b36714bca3fef8ad7b  c6865c523687cb3f235aa599afef1710   
415    56a4ac10a4a8f2ba7693523bb439eede  78438ba6ace7d2cb023dbbc81b083562   
...                                 ...                               ...   
99091  240ead1a7284667e0ec71d01f80e4d5e  fcdd7556401aaa1c980f8b67a69f95dc   
99230  78008d03bd8ef7fcf1568728b316553c  043e3254e68daf7256bda1c9c03c2286   
99266  76a948cd55bf22799753720d4545dd2d  3f20a07b28aa252d0502fe7f7eb030a9   
99377  a6bd1f93b7ff72cc348ca07f38ec4bee  6d63fa86bd2f62908ad328325799152f   
99406  7fd85cb0143de098a4c5ab5a57bfbd91  d32034dfc685b1ae15dd4c78eace868e   

      order_status order_purchase_timestamp   order_approved_at  \
15      

Se identificaron 1359 pedidos en los que la fecha de entrega al transportista es anterior a la fecha de aprobación. Estos registros representan posibles inconsistencias temporales o desfases en el registro de eventos y deberán revisarse antes del análisis.

- La entrega no puede ser antes del envío

In [74]:
print(orders[
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
])

                               order_id                       customer_id  \
6437   a1abeb653a4d4cd1e142ccb8c82cd069  5f50465da00b7fed5dd1239f4ecf6e2c   
9553   383aa8b2724fe452d9ccd9934a8c628b  b1cb2f9d7a19480f3749e248db14d58f   
13487  cb1134f9010d242e9515ad1c78ec0c39  2fd33ac77677bd214b1882868317eeed   
14474  dceb62e8fa94b46006c9554fed743df0  2721900eb4e0f1cc2c836dd7bc1b1e11   
19268  5f9d46795c3126674e52becb3a1a517f  79287bcaafdde5c793b996fc40bb7d9f   
21338  8c78d01de3a9009e23d6877a7cc9be20  6cd7106899e59a1fbd0622d5f1efedf4   
22520  b27af682321527a6349f1761eb3f360c  9859dd92e872dbaa60ca3cd5f0d7ad07   
25393  1cc3ae63caffff2d6c3ee3e78e074acf  01c843a2c0600def0b7693dba47af460   
25646  e37f11cae9985ca58f0b56f268720537  3947a361301f2ff0f3223159a0f2701c   
27470  fa3e37584f4fdb1ded0e0de700dfcb4e  63be4feff10a0b1d85f2cfbf10df9754   
34939  c1e2bf2b7dd3309f2f5356c6b63968fa  e37d47e7eec62f08dc5deecc7d5532d6   
41636  b866af202be0692766081310cd4085e1  d1800078046ed2e5ae1b0792b695c56e   

Se encontraron 23 pedidos cuya fecha de entrega al cliente es anterior a la fecha en que el transportista recibió el pedido. Estos registros constituyen inconsistencias temporales y deberán ser revisados durante la etapa de limpieza.

## Verificación de consistencia en variables categóricas

In [75]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"\n{'='*20} {nombre} {'='*20}")

    # Seleccionar variables categóricas (tipo object)
    columnas_cat = df.select_dtypes(include="object").columns

    for col in columnas_cat:
        print(f"\nVariable: {col}")
        print(f"Número de categorías: {df[col].nunique(dropna=False)}")
        print(df[col].value_counts(dropna=False).head(10))


==================== Customers ====================

Variable: customer_id
Número de categorías: 99441
customer_id
274fa6071e5e17fe303b9748641082c8    1
e5ed7280cd1a3ac2ba29fd6650d8867c    1
c6ece8a5137f3c9c3a3a12302a19a2ac    1
821a7275a08f32975caceff2e08ea262    1
5eef6cce1f34954c9e7004332388ccc7    1
be631308cb609ff74d0e0fb54815e18c    1
a1b5ca506b592bb72d4caadcbfe71385    1
30c96385d694acb8aa2dc0df1770120b    1
b7c889215de76857c7967c1011125d2d    1
c156d63bdfce1d456bd43cf1c4dadfca    1
Name: count, dtype: int64

Variable: customer_unique_id
Número de categorías: 96096
customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
dc813062e0fc23409cd255f7f53c7074     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
de34b16117594161a6a89c50b289d35a     6
63cfc61cee11cbe306bff5857d00bfe4     6
Name: count

Se revisaron las variables categóricas de todas las tablas mediante el análisis del número de categorías y de las frecuencias de cada una. En general, las variables presentan un comportamiento consistente y no se identificaron valores inesperados en campos como order_status, payment_type, customer_state y seller_state, cuyos valores corresponden a las categorías esperadas del dominio.

Se evaluó la consistencia de las bases de datos mediante la revisión de registros duplicados, tipos de datos, valores fuera de rango, secuencia temporal de las fechas y coherencia de las variables categóricas. En general, las tablas presentan un buen nivel de consistencia, ya que la mayoría de los atributos cumplen con el tipo de dato esperado, no se identificaron duplicados en las tablas principales (excepto en geolocation, donde existen 261.831 registros duplicados), y los valores numéricos se encuentran dentro de los rangos definidos para cada variable. Asimismo, las variables categóricas mantienen categorías coherentes con el dominio del negocio, aunque se identificaron diferencias de formato en algunos valores textuales, como el uso de mayúsculas, minúsculas y acentos en nombres de ciudades y comentarios de las reseñas. Adicionalmente, la validación temporal permitió detectar registros cuya secuencia de fechas no sigue el orden esperado del proceso logístico, los cuales serán revisados durante la etapa de limpieza antes de la integración de los datos.

## Trazabilidad de los datos

## Objetivo

Verificar que el conjunto de datos permita rastrear el ciclo completo de una compra mediante la relación entre las tablas, comprobando la integridad de las llaves primarias y foráneas para garantizar la trazabilidad de la información.

In [76]:
print("Customers:", customers["customer_id"].is_unique)
print("Orders:", orders["order_id"].is_unique)
print("Products:", products["product_id"].is_unique)
print("Sellers:", sellers["seller_id"].is_unique)
print("Category Translation:", category_translation["product_category_name"].is_unique)


Customers: True
Orders: True
Products: True
Sellers: True
Category Translation: True


Se comprobó que las llaves primarias de las tablas principales son únicas (True), garantizando que cada registro pueda identificarse de manera individual.

In [77]:
print("Orders -> Customers:",
      orders["customer_id"].isin(customers["customer_id"]).all())

print("Order Items -> Orders:",
      order_items["order_id"].isin(orders["order_id"]).all())

print("Order Items -> Products:",
      order_items["product_id"].isin(products["product_id"]).all())

print("Order Items -> Sellers:",
      order_items["seller_id"].isin(sellers["seller_id"]).all())

print("Order Reviews -> Orders:",
      order_reviews["order_id"].isin(orders["order_id"]).all())

print("Order Payments -> Orders:",
      order_payments["order_id"].isin(orders["order_id"]).all())

Orders -> Customers: True
Order Items -> Orders: True
Order Items -> Products: True
Order Items -> Sellers: True
Order Reviews -> Orders: True
Order Payments -> Orders: True


Los resultados obtenidos muestran que todas las llaves foráneas corresponden a registros válidos en sus respectivas tablas. Esto confirma que las relaciones entre clientes, pedidos, productos, vendedores, pagos y reseñas son consistentes y permiten seguir correctamente el flujo de información.

In [78]:
print("Clientes:", customers.shape[0])
print("Órdenes:", orders.shape[0])
print("Productos:", products.shape[0])
print("Items:", order_items.shape[0])
print("Pagos:", order_payments.shape[0])
print("Reseñas:", order_reviews.shape[0])
print("Vendedores:", sellers.shape[0])

Clientes: 99441
Órdenes: 99441
Productos: 32951
Items: 112650
Pagos: 103886
Reseñas: 99224
Vendedores: 3095


El conjunto de datos está compuesto por 99.441 clientes y 99.441 órdenes, lo que evidencia una correspondencia directa entre ambas tablas. Además, contiene 32.951 productos, 112.650 ítems de pedidos, 103.886 registros de pagos, 99.224 reseñas y 3.095 vendedores, lo que refleja un modelo relacional donde una orden puede incluir múltiples productos, pagos y reseñas asociadas.

In [79]:
import pandas as pd

trazabilidad = pd.DataFrame({
    "Tabla": [
        "Customers",
        "Orders",
        "Order Items",
        "Products",
        "Order Payments",
        "Order Reviews",
        "Sellers"
    ],
    "Llave primaria": [
        "customer_id",
        "order_id",
        "order_id",
        "product_id",
        "order_id",
        "review_id",
        "seller_id"
    ],
    "Se relaciona con": [
        "Orders",
        "Customers, Order Items, Payments, Reviews",
        "Orders, Products, Sellers",
        "Order Items",
        "Orders",
        "Orders",
        "Order Items"
    ],
    "Integridad": [
        "✔",
        "✔",
        "✔",
        "✔",
        "✔",
        "✔",
        "✔"
    ]
})

trazabilidad

,Tabla,Llave primaria,Se relaciona con,Integridad
0,Customers,customer_id,Orders,✔
1,Orders,order_id,"Customers, Order Items, Payments, Reviews",✔
2,Order Items,order_id,"Orders, Products, Sellers",✔
3,Products,product_id,Order Items,✔
4,Order Payments,order_id,Orders,✔
5,Order Reviews,review_id,Orders,✔
6,Sellers,seller_id,Order Items,✔


# Conclusión
La evaluación de la trazabilidad evidenció que el conjunto de datos mantiene relaciones consistentes entre sus tablas mediante llaves primarias y foráneas correctamente definidas. Los resultados obtenidos muestran que la información puede seguirse a lo largo de todo el proceso de compra, desde el cliente hasta el producto, el vendedor, el pago y la reseña. Por ello, el dataset cuenta con una trazabilidad adecuada y una estructura confiable para continuar con el proceso de preparación y análisis de los datos.

# Preparación de los datos

## Imputación de datos faltantes

### Analizar los faltantes en Orders

In [80]:
# Distribución del estado de los pedidos con fecha de aprobación faltante
print(orders[orders["order_approved_at"].isna()]["order_status"].value_counts())

# Fecha de entrega al transportista
print(orders[orders["order_delivered_carrier_date"].isna()]["order_status"].value_counts())

# Fecha de entrega al cliente
print(orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts())

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


### Analizar los faltantes en Products

In [81]:
print(products[products["product_category_name"].isna()])

                             product_id product_category_name  \
105    a41e356c76fab66334f36de622ecbd3a                   NaN   
128    d8dee61c2034d6d075997acef1870e9b                   NaN   
145    56139431d72cd51f19eb9f7dae4d1617                   NaN   
154    46b48281eb6d663ced748f324108c733                   NaN   
197    5fb61f482620cb672f5e586bb132eae9                   NaN   
...                                 ...                   ...   
32515  b0a0c5dd78e644373b199380612c350a                   NaN   
32589  10dbe0fbaa2c505123c17fdc34a63c56                   NaN   
32616  bd2ada37b58ae94cc838b9c0569fecd8                   NaN   
32772  fa51e914046aab32764c41356b9d4ea4                   NaN   
32852  c4ceee876c82b8328e9c293fa0e1989b                   NaN   

       product_name_lenght  product_description_lenght  product_photos_qty  \
105                    NaN                         NaN                 NaN   
128                    NaN                         NaN         

In [82]:
print(products[products["product_category_name"].isna()].isna().sum())

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64


### Analizar Order Reviews

In [83]:
# Comentario faltante según la calificación
print(order_reviews.groupby(
    order_reviews["review_comment_message"].isna()
)["review_score"].value_counts())

review_comment_message  review_score
False                   5               20554
                        1                8745
                        4                5976
                        3                3557
                        2                2145
True                    5               36774
                        4               13166
                        3                4622
                        1                2679
                        2                1006
Name: count, dtype: int64


| **Tabla**         | **Variable**                                                                     | **% de nulos** | **Evidencia encontrada**                                                                                                       | **Clasificación** | **Justificación**                                                                                                                                                     | **Acción propuesta**                                                                                                                               |
| ----------------- | -------------------------------------------------------------------------------- | -------------: | ------------------------------------------------------------------------------------------------------------------------------ | ----------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Orders**        | `order_approved_at`                                                              |      **0.16%** | Los faltantes se concentran principalmente en pedidos `canceled` (141), además de 14 `delivered` y 5 `created`.                | **MAR**           | La ausencia depende del estado del pedido (`order_status`). En los pedidos entregados la aprobación necesariamente ocurrió, aunque no fue registrada.                 | **Imputar únicamente los 14 pedidos `delivered`** utilizando la mediana del tiempo entre la compra y la aprobación. Mantener los demás como nulos. |
| **Orders**        | `order_delivered_carrier_date`                                                   |      **1.79%** | La mayoría corresponde a pedidos `unavailable`, `canceled`, `processing` e `invoiced`; solo 2 son `delivered`.                 | **MAR**           | La ausencia está relacionada con el flujo logístico del pedido. No existe evidencia suficiente para reconstruir la fecha exacta.                                      | Mantener como nulo y revisar manualmente los 2 casos `delivered`.                                                                                  |
| **Orders**        | `order_delivered_customer_date`                                                  |      **2.98%** | Los faltantes pertenecen principalmente a pedidos `shipped`, `processing`, `canceled` y `unavailable`. Solo 8 son `delivered`. | **MAR**           | Es esperable que los pedidos no entregados no tengan registrada esta fecha. En los pedidos `delivered` no es posible conocer la fecha real sin información adicional. | Mantener como nulo y revisar los casos `delivered`; no imputar automáticamente.                                                                    |
| **Products**      | `product_category_name`                                                          |      **1.85%** | Los mismos 610 registros presentan también ausencia en nombre, descripción y fotos.                                            | **MNAR**          | La ausencia hace parte de un mismo bloque de información descriptiva del producto.                                                                                    | Mantener como nulo o crear la categoría **"Unknown"** si el análisis requiere una categoría para todos los productos.                              |
| **Products**      | `product_name_lenght`                                                            |      **1.85%** | Coincide exactamente con los mismos 610 productos sin información descriptiva.                                                 | **MNAR**          | No existe información suficiente para reconstruir el valor real.                                                                                                      | Mantener como nulo o excluir estos registros si la variable será utilizada en el análisis.                                                         |
| **Products**      | `product_description_lenght`                                                     |      **1.85%** | Coincide exactamente con los mismos 610 productos sin información descriptiva.                                                 | **MNAR**          | Hace parte del mismo bloque de atributos faltantes.                                                                                                                   | Mantener como nulo o excluir estos registros si la variable será utilizada.                                                                        |
| **Products**      | `product_photos_qty`                                                             |      **1.85%** | Coincide exactamente con los mismos 610 registros.                                                                             | **MNAR**          | La ausencia depende de la falta de información descriptiva del producto.                                                                                              | Mantener como nulo. Solo asignar **0** si el contexto del proyecto lo justifica explícitamente.                                                    |
| **Products**      | `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` |      **0.01%** | Solo 2 registros presentan valores faltantes.                                                                                  | **MCAR**          | No se observa un patrón evidente; sin embargo, no existe información suficiente para estimar de forma confiable las características físicas del producto.             | **Mantener los valores nulos o eliminar únicamente esos dos registros** si estas variables serán utilizadas en el análisis.                        |
| **Order Reviews** | `review_comment_title`                                                           |     **88.34%** | La mayoría de los clientes calificó la compra sin escribir un título.                                                          | **MNAR**          | Es una decisión voluntaria del cliente dejar o no un título.                                                                                                          | Mantener como nulo o reemplazar por **"Sin título"** únicamente para análisis de texto.                                                            |
| **Order Reviews** | `review_comment_message`                                                         |     **58.70%** | Muchos usuarios califican la compra sin escribir un comentario.                                                                | **MNAR**          | La ausencia depende de la decisión del cliente de escribir un comentario.                                                                                             | Mantener como nulo o reemplazar por **"Sin comentario"** únicamente para análisis de texto.                                                        |


### Verificar casos de delivered

In [84]:
print(orders[
    (orders["order_status"]=="delivered") &
    (orders["order_approved_at"].isna())
])

                               order_id                       customer_id  \
5323   e04abd8149ef81b95221e88f6ed9ab6a  2127dc6603ac33544953ef05ec155771   
16567  8a9adc69528e1001fc68dd0aaebbb54a  4c1ccc74e00993733742a3c786dc3c1f   
19031  7013bcfc1c97fe719a7b5e05e61c12db  2941af76d38100e0f8740a374f1a5dc3   
22663  5cf925b116421afa85ee25e99b4c34fb  29c35fc91fc13fb5073c8f30505d860d   
23156  12a95a3c06dbaec84bcfb0e2da5d228a  1e101e0daffaddce8159d25a8e53f2b2   
26800  c1d4211b3dae76144deccd6c74144a88  684cb238dc5b5d6366244e0e0776b450   
38290  d69e5d356402adc8cf17e08b5033acfb  68d081753ad4fe22fc4d410a9eb1ca01   
39334  d77031d6a3c8a52f019764e68f211c69  0bf35cac6cc7327065da879e2d90fae8   
48401  7002a78c79c519ac54022d4f8a65e6e8  d5de688c321096d15508faae67a27051   
61743  2eecb0d85f281280f79fa00f9cec1a95  a3d3c38e58b9d2dfb9207cab690b6310   
63052  51eb2eebd5d76a24625b31c33dd41449  07a2a7e0f63fd8cb757ed77d4245623c   
67697  88083e8f64d95b932164187484d90212  f67cd1a215aae2a1074638bbd35a223a   

### Tiempo trancurrido entre la compra y la aprobación

### Imputar order_approved_at

In [85]:
tiempo_aprobacion = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds()/3600

print(tiempo_aprobacion.describe())

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
dtype: float64


### Verificar datos extremos de días

In [86]:
dias_aprobacion = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [87]:
print(orders.assign(
    dias_aprobacion=dias_aprobacion
).sort_values("dias_aprobacion", ascending=False).head(20))

                               order_id                       customer_id  \
47552  1612081119e8f23745698ad3367cc14b  20d32833d8983a835cafcd54099631a0   
62293  2e5dc86c8c4aa663549caf5e31de840d  6225eed02b7d1a110b6e5b5dd4c8bd31   
4541   2e7a8482f6fb09756ca50c10d7bfc047  08c5351a6aca1c1589a38f244edeee9d   
4396   e5fa5a7210941f7d56d0208e4e071d35  683c54fc24d40ee9f8a6fc179fd9856c   
43697  0a5c74ccc786ced7903270de9d6c170a  c021456db05f8e71f0985bef8859b793   
96251  0a93b40850d3f4becf2f276666e01340  a70076d8d4bfce15f8081951c43bf187   
55708  f7923db0430587601c2aef15ec4b8af4  1318fd61a82a479edaff00217376c052   
53475  490291524fddde2b31c2e6bec3d9e6da  ea2ba2a668051e4b78923fc0ef607f86   
10071  809a282bbd5dbcabb6f2f724fca862ec  622e13439d6b5a0b486c435618b2679e   
83143  fdd647b689626410b725d1cce2ddf37c  3714eb406c7704a920e504085717ea5b   
88671  de0076b42a023f53b398ce9ab0d9009c  9c48b4a1e7be90b187665db74770750b   
71651  daed0f3aefd193de33c31e21b16a3b3a  cf258f4e485365461f6a68cf2720c60e   

La variable presenta una distribución altamente asimétrica debido a la existencia de valores extremos, con tiempos de aprobación de hasta 188 días. Estos registros corresponden principalmente a pedidos cancelados, no disponibles o en procesamiento, por lo que no representan el comportamiento habitual del proceso de compra. En consecuencia, se utilizó la mediana del tiempo entre la compra y la aprobación para imputar los pedidos entregados con fecha de aprobación faltante, ya que esta medida es robusta frente a valores atípicos.

### Calcular la mediana de días para pedidos entregados para imputar los faltantes

In [88]:
dias = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds()/86400

dias_delivered = dias[
    (orders["order_status"]=="delivered") &
    (dias.notna())
]

print(dias_delivered.describe())

count    96464.000000
mean         0.428199
std          0.855642
min          0.000000
25%          0.008970
50%          0.014306
75%          0.604774
max         30.893484
dtype: float64


El tiempo transcurrido entre la compra y la aprobación del pedido presenta una distribución asimétrica positiva. Aunque el promedio es de 0.43 días es decir aproximadamente 10 horas, la mediana es de solo 0.014 días es decir 20.6 minutos, lo que indica que la mayoría de las aprobaciones ocurren poco después de la compra y que existen algunos casos excepcionales con demoras de hasta 30.89 días.

In [89]:
mediana = dias_aprobacion.median()
mediana

np.float64(0.014305555555555556)

In [90]:
orders.loc[
    (orders["order_approved_at"].isna()) &
    (orders["order_status"] == "delivered"),
    "order_approved_at"
] = (
    orders["order_purchase_timestamp"] +
    pd.to_timedelta(mediana, unit="D")
)

Se identificaron 14 pedidos con estado delivered que no registraban la fecha de aprobación (order_approved_at). Dado que estos pedidos fueron enviados y entregados, la aprobación debió ocurrir, por lo que la ausencia de la fecha se interpreta como un problema de registro y no como la inexistencia del evento. El tiempo entre la compra y la aprobación presenta una mediana de aproximadamente 20 minutos, mientras que la media se encuentra influenciada por valores atípicos. Por esta razón, los valores faltantes fueron imputados utilizando la mediana del tiempo transcurrido entre la compra y la aprobación, preservando el comportamiento típico del proceso y evitando el efecto de los valores extremos.

### Análisis de los faltantes en order_delivered_carrier_date y order_delivered_customer_date

In [91]:
print(orders[
    (orders["order_status"]=="delivered") &
    (orders["order_delivered_carrier_date"].isna())
])

                               order_id                       customer_id  \
73222  2aa91108853cecb43c84a5dc5b277475  afeb16c7f46396c0ed54acb45ccaaa40   
92643  2d858f451373b04fb5c984a1cc2defaf  e08caf668d499a6d643dafd7c5cc498a   

      order_status order_purchase_timestamp   order_approved_at  \
73222    delivered      2017-09-29 08:52:58 2017-09-29 09:07:16   
92643    delivered      2017-05-25 23:22:43 2017-05-25 23:30:16   

      order_delivered_carrier_date order_delivered_customer_date  \
73222                          NaT           2017-11-20 19:44:47   
92643                          NaT                           NaT   

      order_estimated_delivery_date  
73222                    2017-11-14  
92643                    2017-06-23  


In [92]:
print(orders[
    (orders["order_status"]=="delivered") &
    (orders["order_delivered_customer_date"].isna())
])

                               order_id                       customer_id  \
3002   2d1e2d5bf4dc7227b3bfebb81328c15f  ec05a6d8558c6455f0cbbd8a420ad34f   
20618  f5dd62b788049ad9fc0526e3ad11a097  5e89028e024b381dc84a13a3570decb4   
43834  2ebdfc4f15f23b91474edf87475f108e  29f0540231702fda0cfdee0a310f11aa   
79263  e69f75a717d64fc5ecdfae42b2e8e086  cfda40ca8dd0a5d486a9635b611b398a   
82868  0d3268bad9b086af767785e3f0fc0133  4f1d63d35fb7c8999853b2699f5c7649   
92643  2d858f451373b04fb5c984a1cc2defaf  e08caf668d499a6d643dafd7c5cc498a   
97647  ab7c89dc1bf4a1ead9d6ec1ec8968a84  dd1b84a7286eb4524d52af4256c0ba24   
98038  20edc82cf5400ce95e1afacc25798b31  28c37425f1127d887d7337f284080a0f   

      order_status order_purchase_timestamp   order_approved_at  \
3002     delivered      2017-11-28 17:44:07 2017-11-28 17:56:40   
20618    delivered      2018-06-20 06:58:43 2018-06-20 07:19:05   
43834    delivered      2018-07-01 17:05:11 2018-07-01 17:15:12   
79263    delivered      2018-07-01 22:

En esta parte podemos ver dos casos especificos los cuales son:
* Caso 73222: Este pedido sí fue entregado, por lo que es lógico asumir que el transportista necesariamente recibió el paquete antes de entregarlo al cliente.

En este caso se podría imputar usando una estimación basada en el comportamiento normal de los pedidos

In [93]:
dias_envio = (
    orders["order_delivered_customer_date"] -
    orders["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

print(dias_envio.describe())

count    96475.000000
mean         9.330547
std          8.760122
min        -16.096169
25%          4.099948
50%          7.099769
75%         12.029115
max        205.190972
dtype: float64


Se calculó la distribución del tiempo entre la fecha de entrega al transportista y la fecha de entrega al cliente utilizando los 96.464 pedidos que contienen ambas fechas. La distribución presenta una fuerte asimetría positiva debido a algunos valores extremos de hasta 30.89 días, mientras que la mediana es de apenas 0.014 días de apóximadamente 20 minutos. Debido a esta asimetría, la mediana constituye una medida más robusta que la media para representar el comportamiento típico del proceso y será utilizada para imputar los registros faltantes.

In [94]:
orders.loc[73222, "order_delivered_carrier_date"] = (
    orders.loc[73222, "order_delivered_customer_date"] -
    pd.Timedelta(days=dias_envio.median())
)

El otro caso es 92643 sin embargo en este caso no se puede imputar ya que el pedido no fue entregado, por lo que no se puede asumir que el transportista recibió el paquete. Entonces lo dejarmeos como nulo y no se imputará.

Aunque existen pedidos con estado delivered sin fecha de entrega al cliente, esta variable representa un evento observado y no puede inferirse con certeza a partir de otras variables. Imputar una fecha utilizando la mediana del tiempo de envío implicaría asumir que el pedido fue entregado en un momento específico sin evidencia que lo respalde. Dado que únicamente se identificaron ocho registros (menos del 0.01 % del conjunto de datos), se decidió conservar estos valores como faltantes o excluir dichos registros únicamente en los análisis que requieran la fecha exacta de entrega.

## Atípicos

In [95]:
import pandas as pd

def detectar_outliers_iqr(df, nombre):
    print(f"\n===== {nombre} =====")

    columnas = df.select_dtypes(include="number").columns

    for col in columnas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        limite_inf = Q1 - 1.5 * IQR
        limite_sup = Q3 + 1.5 * IQR

        n_outliers = ((df[col] < limite_inf) | (df[col] > limite_sup)).sum()
        porcentaje = n_outliers / len(df) * 100

        print(f"{col}")
        print(f"  Outliers: {n_outliers}")
        print(f"  Porcentaje: {porcentaje:.2f}%")
        print(f"  Límite inferior: {limite_inf:.2f}")
        print(f"  Límite superior: {limite_sup:.2f}")
        print("-"*50)

In [96]:
detectar_outliers_iqr(products, "Products")

detectar_outliers_iqr(order_items, "Order Items")

detectar_outliers_iqr(order_payments, "Order Payments")

detectar_outliers_iqr(order_reviews, "Order Reviews")


===== Products =====
product_name_lenght
  Outliers: 290
  Porcentaje: 0.88%
  Límite inferior: 19.50
  Límite superior: 79.50
--------------------------------------------------
product_description_lenght
  Outliers: 2078
  Porcentaje: 6.31%
  Límite inferior: -610.50
  Límite superior: 1921.50
--------------------------------------------------
product_photos_qty
  Outliers: 849
  Porcentaje: 2.58%
  Límite inferior: -2.00
  Límite superior: 6.00
--------------------------------------------------
product_weight_g
  Outliers: 4551
  Porcentaje: 13.81%
  Límite inferior: -2100.00
  Límite superior: 4300.00
--------------------------------------------------
product_length_cm
  Outliers: 1380
  Porcentaje: 4.19%
  Límite inferior: -12.00
  Límite superior: 68.00
--------------------------------------------------
product_height_cm
  Outliers: 1892
  Porcentaje: 5.74%
  Límite inferior: -11.50
  Límite superior: 40.50
--------------------------------------------------
product_width_cm
  Out

| Tabla              | Variable                     | % de outliers | ¿Son realmente un problema?                                            | Acción recomendada                                 |
| ------------------ | ---------------------------- | ------------: | ---------------------------------------------------------------------- | -------------------------------------------------- |
| **Products**       | `product_name_lenght`        |         0.88% | No necesariamente. Existen nombres muy largos.                         | Mantener.                                          |
| **Products**       | `product_description_lenght` |         6.31% | No. Algunas descripciones son naturalmente extensas.                   | Mantener.                                          |
| **Products**       | `product_photos_qty`         |         2.58% | No. Algunos vendedores publican muchas fotos.                          | Mantener.                                          |
| **Products**       | `product_weight_g`           |        13.81% | Revisar. Puede haber productos realmente pesados o errores de captura. | Inspeccionar los valores máximos antes de decidir. |
| **Products**       | `product_length_cm`          |         4.19% | Revisar. Puede haber productos grandes.                                | Inspeccionar.                                      |
| **Products**       | `product_height_cm`          |         5.74% | Revisar.                                                               | Inspeccionar.                                      |
| **Products**       | `product_width_cm`           |         2.77% | Revisar.                                                               | Inspeccionar.                                      |
| **Order Items**    | `order_item_id`              |        12.41% | **No aplica.** Es un identificador secuencial dentro del pedido.       | No analizar como outlier.                          |
| **Order Items**    | `price`                      |         7.48% | No necesariamente. Existen productos de alto precio.                   | Revisar extremos, mantener si son coherentes.      |
| **Order Items**    | `freight_value`              |        10.77% | No necesariamente. El flete depende del peso y la distancia.           | Revisar extremos.                                  |
| **Order Payments** | `payment_sequential`         |         4.36% | **No aplica.** Es el número de pago realizado por un cliente.          | No analizar como outlier.                          |
| **Order Payments** | `payment_installments`       |         6.08% | No necesariamente. Algunos clientes pagan en muchas cuotas.            | Mantener salvo valores imposibles.                 |
| **Order Payments** | `payment_value`              |         7.68% | No necesariamente. Existen compras de alto valor.                      | Revisar extremos.                                  |
| **Order Reviews**  | `review_score`               |        14.69% | **No aplica.** Es una variable ordinal de 1 a 5.                       | No realizar tratamiento de outliers.               |


### Inspección de variables

In [97]:
print("===== Products =====")
print("\nproduct_weight_g")
print(products["product_weight_g"].describe())

print("\nproduct_length_cm")
print(products["product_length_cm"].describe())

print("\nproduct_height_cm")
print(products["product_height_cm"].describe())

print("\nproduct_width_cm")
print(products["product_width_cm"].describe())


print("\n===== Order Items =====")
print("\nprice")
print(order_items["price"].describe())

print("\nfreight_value")
print(order_items["freight_value"].describe())


print("\n===== Order Payments =====")
print("\npayment_value")
print(order_payments["payment_value"].describe())

print("\npayment_installments")
print(order_payments["payment_installments"].describe())

===== Products =====

product_weight_g
count    32949.000000
mean      2276.472488
std       4282.038731
min          0.000000
25%        300.000000
50%        700.000000
75%       1900.000000
max      40425.000000
Name: product_weight_g, dtype: float64

product_length_cm
count    32949.000000
mean        30.815078
std         16.914458
min          7.000000
25%         18.000000
50%         25.000000
75%         38.000000
max        105.000000
Name: product_length_cm, dtype: float64

product_height_cm
count    32949.000000
mean        16.937661
std         13.637554
min          2.000000
25%          8.000000
50%         13.000000
75%         21.000000
max        105.000000
Name: product_height_cm, dtype: float64

product_width_cm
count    32949.000000
mean        23.196728
std         12.079047
min          6.000000
25%         15.000000
50%         20.000000
75%         30.000000
max        118.000000
Name: product_width_cm, dtype: float64

===== Order Items =====

price
count    11

El método del rango intercuartílico (IQR) identificó valores atípicos en varias variables numéricas. Sin embargo, tras revisar sus estadísticas descriptivas, se observó que los valores máximos corresponden a situaciones plausibles del dominio del comercio electrónico (productos pesados, artículos de alto precio, costos de envío elevados y pagos en múltiples cuotas). No se identificaron valores imposibles o inconsistentes, por lo que se decidió mantener estos registros, ya que representan la variabilidad natural de los datos y su eliminación podría introducir sesgos en el análisis posterior.

### Areglar inconsistencia 

Se detectaron inconsistencias en las fechas de entrega, donde el cliente aparecía recibiendo el pedido antes de que el transportista lo recibiera. Al no ser posible identificar cuál fecha era la correcta, se optó por reemplazar ambas variables (order_delivered_carrier_date y order_delivered_customer_date) por NaN, preservando la integridad del conjunto de datos y evitando introducir supuestos sin evidencia.

In [98]:
# Identificar pedidos donde la fecha de entrega al cliente
# es anterior a la fecha en que el transportista recibió el pedido
mask = (
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
)

# Ver cuántos casos existen
print(f"Pedidos inconsistentes: {mask.sum()}")

# Reemplazar ambas fechas por NaT
orders.loc[
    mask,
    ["order_delivered_carrier_date", "order_delivered_customer_date"]
] = pd.NaT

# Verificar que ya no existan inconsistencias
mask_verificacion = (
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
)

print(f"Inconsistencias restantes: {mask_verificacion.sum()}")

Pedidos inconsistentes: 23
Inconsistencias restantes: 0


Revisar el caso de  1359 pedidos en los que la fecha de entrega al transportista es anterior a la fecha de aprobación. Estos registros representan posibles inconsistencias temporales o desfases en el registro de eventos para ver si es una inconsistencia o si se encuentra un patrón en los datos.

- Revisar cuanto tiempo de difernecia existe 

In [99]:
dias_diferencia = (
    orders["order_approved_at"] -
    orders["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

mask = orders["order_delivered_carrier_date"] < orders["order_approved_at"]

print(dias_diferencia[mask].describe())

count    1359.000000
mean        1.031333
std         4.804491
min         0.000243
25%         0.058976
50%         0.715324
75%         1.081528
max       171.219005
dtype: float64


Se identificaron 1359 pedidos en los que la fecha de entrega al transportista es anterior a la fecha de aprobación. No obstante, el análisis de la magnitud de la diferencia mostró que el 75 % de los casos presenta desfases inferiores a 1.1 días (mediana de 0.72 días), lo que sugiere retrasos en el registro de los eventos más que errores en el proceso logístico. Debido a ello, estos registros se conservaron, ya que no existe evidencia suficiente para corregir o eliminar las fechas. Solo los casos con diferencias extremadamente altas serán revisados individualmente por considerarse posibles errores de captura.

In [100]:
# Diferencia en días entre la aprobación y la entrega al transportista
dias_diferencia = (
    orders["order_approved_at"] -
    orders["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

# Casos con diferencias mayores a 7 días
casos_extremos = orders.loc[dias_diferencia > 7].copy()

# Agregar la diferencia en días
casos_extremos["dias_diferencia"] = dias_diferencia[dias_diferencia > 7]

# Número de casos
print(f"Número de casos extremos: {len(casos_extremos)}\n")

# Distribución por estado del pedido
print("Distribución por estado del pedido:")
print(casos_extremos["order_status"].value_counts())

print("\nDetalle de los casos extremos:")
print(
    casos_extremos[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "dias_diferencia"
        ]
    ].sort_values("dias_diferencia", ascending=False)
)

Número de casos extremos: 14

Distribución por estado del pedido:
order_status
delivered    14
Name: count, dtype: int64

Detalle de los casos extremos:
                               order_id order_status order_purchase_timestamp  \
25883  7c48bb55e8e4f7e56d412e9653db37bc    delivered      2018-07-16 18:40:53   
14562  1fab4ac9d85079b3da72a11475ae1685    delivered      2017-09-01 19:04:22   
46163  0184d4ddb259e1a4cfc2871888cf97b8    delivered      2017-09-01 20:04:28   
98710  1378f9601350615613cc8832d6789c5d    delivered      2017-09-01 20:28:02   
41592  8554cb37f7158cb0b082a841d24a4589    delivered      2017-09-01 18:40:44   
11738  cf72398d0690f841271b695bbfda82d2    delivered      2017-09-01 18:45:33   
85393  40de47dfa620d667117e4a6067b6e1ec    delivered      2017-09-01 20:05:55   
31211  bc4854efd86d9f42140c951c595d20c1    delivered      2017-09-01 20:05:42   
55302  77ca435b03fbf991e5027e3776e37885    delivered      2017-09-01 18:49:54   
68315  580f3268bc2075c5961b2d2929c7a3

- Revisar si las fechas anteriores son coherentes con las fechas de entrega al cliente

In [101]:
casos_extremos["dias_carrier_cliente"] = (
    casos_extremos["order_delivered_customer_date"] -
    casos_extremos["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

print(casos_extremos[[
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "dias_diferencia",
    "dias_carrier_cliente"
]])

                               order_id order_purchase_timestamp  \
11738  cf72398d0690f841271b695bbfda82d2      2017-09-01 18:45:33   
14562  1fab4ac9d85079b3da72a11475ae1685      2017-09-01 19:04:22   
25883  7c48bb55e8e4f7e56d412e9653db37bc      2018-07-16 18:40:53   
27309  c3b8c17ee15e0e798c2e178b7d4c7f42      2017-09-01 20:04:47   
31211  bc4854efd86d9f42140c951c595d20c1      2017-09-01 20:05:42   
41592  8554cb37f7158cb0b082a841d24a4589      2017-09-01 18:40:44   
46163  0184d4ddb259e1a4cfc2871888cf97b8      2017-09-01 20:04:28   
55090  06eb87385425e5797a1a5c2cdb1b6559      2017-09-01 18:53:08   
55302  77ca435b03fbf991e5027e3776e37885      2017-09-01 18:49:54   
66460  f6f0b2497c5a4ca89670186757ab2684      2017-06-26 12:23:32   
68315  580f3268bc2075c5961b2d2929c7a35b      2017-09-01 20:17:09   
68417  70f357cca87c1162357bf3c0a993cbe5      2017-09-01 18:40:11   
85393  40de47dfa620d667117e4a6067b6e1ec      2017-09-01 20:05:55   
98710  1378f9601350615613cc8832d6789c5d      201

- Cuanto tardó desde la compra hasta la entrega al cliente

In [102]:
casos_extremos["dias_compra_cliente"] = (
    casos_extremos["order_delivered_customer_date"] -
    casos_extremos["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(casos_extremos[[
    "order_id",
    "dias_compra_cliente"
]].sort_values("dias_compra_cliente", ascending=False))

                               order_id  dias_compra_cliente
55302  77ca435b03fbf991e5027e3776e37885            28.947095
68315  580f3268bc2075c5961b2d2929c7a35b            16.899433
55090  06eb87385425e5797a1a5c2cdb1b6559            14.000486
98710  1378f9601350615613cc8832d6789c5d            12.081065
68417  70f357cca87c1162357bf3c0a993cbe5            10.926042
27309  c3b8c17ee15e0e798c2e178b7d4c7f42             9.860405
11738  cf72398d0690f841271b695bbfda82d2             9.812141
46163  0184d4ddb259e1a4cfc2871888cf97b8             7.797407
66460  f6f0b2497c5a4ca89670186757ab2684             7.229132
41592  8554cb37f7158cb0b082a841d24a4589             7.060428
25883  7c48bb55e8e4f7e56d412e9653db37bc             7.058241
14562  1fab4ac9d85079b3da72a11475ae1685             7.047697
85393  40de47dfa620d667117e4a6067b6e1ec             7.024549
31211  bc4854efd86d9f42140c951c595d20c1             5.090312


Se identificaron 1.359 registros en los que la fecha de aprobación era posterior a la fecha de entrega al transportista, lo cual contradice el flujo esperado del proceso logístico. Tras revisar los casos más extremos, se observó que la secuencia entre compra, envío y entrega al cliente era consistente, mientras que la fecha de aprobación era la única que rompía el orden temporal. Dado que no es posible determinar la fecha real de aprobación sin una fuente externa, se decidió reemplazar estos valores por NaT para evitar introducir información artificial y preservar la calidad temporal del conjunto de datos.

In [103]:
# Casos donde la aprobación ocurre después de la entrega al transportista
mask = orders["order_approved_at"] > orders["order_delivered_carrier_date"]

# Se considera la fecha de aprobación inconsistente
orders.loc[mask, "order_approved_at"] = pd.NaT

### Revisión de limpieza de imputación y outliers 

In [104]:
print(orders[

    orders["order_delivered_carrier_date"] < orders["order_approved_at"]

])
print(orders[

    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]

])

Empty DataFrame
Columns: [order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]
Index: []
Empty DataFrame
Columns: [order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]
Index: []


In [105]:
detectar_outliers_iqr(products, "Products")

detectar_outliers_iqr(order_items, "Order Items")

detectar_outliers_iqr(order_payments, "Order Payments")

detectar_outliers_iqr(order_reviews, "Order Reviews")


===== Products =====
product_name_lenght
  Outliers: 290
  Porcentaje: 0.88%
  Límite inferior: 19.50
  Límite superior: 79.50
--------------------------------------------------
product_description_lenght
  Outliers: 2078
  Porcentaje: 6.31%
  Límite inferior: -610.50
  Límite superior: 1921.50
--------------------------------------------------
product_photos_qty
  Outliers: 849
  Porcentaje: 2.58%
  Límite inferior: -2.00
  Límite superior: 6.00
--------------------------------------------------
product_weight_g
  Outliers: 4551
  Porcentaje: 13.81%
  Límite inferior: -2100.00
  Límite superior: 4300.00
--------------------------------------------------
product_length_cm
  Outliers: 1380
  Porcentaje: 4.19%
  Límite inferior: -12.00
  Límite superior: 68.00
--------------------------------------------------
product_height_cm
  Outliers: 1892
  Porcentaje: 5.74%
  Límite inferior: -11.50
  Límite superior: 40.50
--------------------------------------------------
product_width_cm
  Out

payment_sequential
  Outliers: 4526
  Porcentaje: 4.36%
  Límite inferior: 1.00
  Límite superior: 1.00
--------------------------------------------------
payment_installments
  Outliers: 6313
  Porcentaje: 6.08%
  Límite inferior: -3.50
  Límite superior: 8.50
--------------------------------------------------
payment_value
  Outliers: 7981
  Porcentaje: 7.68%
  Límite inferior: -115.78
  Límite superior: 344.41
--------------------------------------------------

===== Order Reviews =====
review_score
  Outliers: 14575
  Porcentaje: 14.69%
  Límite inferior: 2.50
  Límite superior: 6.50
--------------------------------------------------


Cómo podemos ver ya no se encuentran las inconsistencias en las fechas de entrega, ya que se reemplazaron por NaN. Además, los valores atípicos identificados en las variables numéricas fueron revisados y se determinó que corresponden a situaciones plausibles del dominio, por lo que se decidió mantenerlos para preservar la variabilidad natural de los datos.

In [106]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"\n===== {nombre} =====")
    print(completitud(df))


===== Customers =====
                          Valores no nulos  Valores nulos  \
customer_id                          99441              0   
customer_unique_id                   99441              0   
customer_zip_code_prefix             99441              0   
customer_city                        99441              0   
customer_state                       99441              0   

                          Porcentaje de nulos (%)  
customer_id                                   0.0  
customer_unique_id                            0.0  
customer_zip_code_prefix                      0.0  
customer_city                                 0.0  
customer_state                                0.0  

===== Geolocation =====
                             Valores no nulos  Valores nulos  \
geolocation_zip_code_prefix           1000163              0   
geolocation_lat                       1000163              0   
geolocation_lng                       1000163              0   
geolocation_city 

Cómo podemos ver en cuanto a los faltantes en algunos casos ya se quitaron por completo, pero en otros aumentaron ya que cómo medida para las inconsistencias en las fechas de entrega se reemplazaron por NaN, lo que aumentó la cantidad de faltantes en esas variables.